In [ ]:
%pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub diffusers

In [ ]:
import torch
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv(override=True)
hf_token = os.getenv('HF_API_KEY')
login(hf_token, add_to_git_credential=True)

In [ ]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="mps")
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="mps")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
ner = pipeline("ner", device="mps")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
  print(entity)

In [ ]:
question="What are Hugging Face pipelines?"
context="Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device="mps")
result = question_answerer(question=question, context=context)
print(result)

In [ ]:
import time
import statistics

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
text = "I really enjoyed this movie!"
num_runs = 5

def benchmark_classifier(device):
    classifier = pipeline("text-classification", model=model_name, device=device)

    # Warm up once so model initialization and first-run overhead are excluded.
    classifier(text)
    if device == "mps":
        torch.mps.synchronize()

    run_times = []
    for _ in range(num_runs):
        if device == "mps":
            torch.mps.synchronize()
        start_time = time.perf_counter()
        result = classifier(text)
        if device == "mps":
            torch.mps.synchronize()
        run_times.append(time.perf_counter() - start_time)

    return result, statistics.mean(run_times)

cpu_result, cpu_seconds = benchmark_classifier("cpu")
print(f"CPU result: {cpu_result}")
print(f"CPU average ({num_runs} runs): {cpu_seconds:.4f} s")

if torch.backends.mps.is_available():
    mps_result, mps_seconds = benchmark_classifier("mps")
    print(f"MPS result: {mps_result}")
    print(f"MPS average ({num_runs} runs): {mps_seconds:.4f} s")
    print(f"MPS speedup vs CPU: {cpu_seconds / mps_seconds:.2f}x")
else:
    print("MPS is not available in this PyTorch environment.")

In [ ]:
import time
from transformers import pipeline

start = time.perf_counter()

classifier_mps = pipeline(
    "text-classification",
    device="mps"
)

elapsed = time.perf_counter() - start
print(f"mps: Pipeline initialization took {elapsed:.2f} seconds")

start = time.perf_counter()

classifier_cpu = pipeline(
    "text-classification",
    device="cpu"
)

elapsed = time.perf_counter() - start
print(f"cpu: Pipeline initialization took {elapsed:.2f} seconds")

In [ ]:
result = classifier_mps("I really enjoyed this movie!")
print(result)

result = classifier_cpu("I really enjoyed this movie!")
print(result)

In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device="mps"
)

text = "Tim Cook works at Apple in Cupertino, California."

results = ner(text)

for item in results:
    print(
        f"{item['word']:15} "
        f"{item['entity_group']:5} "
        f"{item['score']:.3f}"
    )

In [ ]:
from transformers import pipeline

question_answerer = pipeline(
    "question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad",
    device="mps"
)

context = """
Apple was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.
The company is headquartered in Cupertino, California.
Its products include the iPhone, Mac, and iPad.
"""

question = "Where is Apple headquartered?"

result = question_answerer(
    question=question,
    context=context
)

print("Answer:", result["answer"])
print("Confidence:", f"{result['score']:.3f}")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="openai-community/gpt2",
    device="mps"
)

prompt = "Artificial intelligence will change the world because"

results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    temperature=0.8,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    temperature=0.8,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.00000001,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.00000001,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
results = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.9999999,
    num_return_sequences=1
)

print(results[0]["generated_text"])

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device="mps"
)

text = """
Artificial intelligence is increasingly being used in software development.
AI tools can help programmers write code, find bugs, generate tests,
and understand unfamiliar codebases. These tools can improve productivity,
but their output is not always correct. Developers still need to review
generated code, test it carefully, and consider security and maintainability.
AI is most effective when it assists developers rather than replacing
engineering judgment.
"""

results = summarizer(
    text,
    max_new_tokens=60,
    min_new_tokens=15,
    do_sample=False
)

print(results[0]["summary_text"])

In [ ]:
from transformers import pipeline

translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr",
    device="mps"
)

text = "Artificial intelligence is changing the way we develop software."

results = translator(text, max_new_tokens=100)

print(results[0]["translation_text"])

In [ ]:
from transformers import pipeline

translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-zh",
    device="mps"
)

text = "Artificial intelligence is changing the way we develop software."

results = translator(text, max_new_tokens=100)

print(results[0]["translation_text"])

In [ ]:
from transformers import pipeline

translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-zh-en",
    device="mps"
)

text = "人工智能正在改变我们开发软件的方式"

results = translator(text, max_new_tokens=100)

print(results[0]["translation_text"])

In [ ]:
text = """
《岳阳楼记》全文
北宋·范仲淹
庆历四年春，滕子京谪守巴陵郡。越明年，政通人和，百废具兴。乃重修岳阳楼，增其旧制，刻唐贤今人诗赋于其上，属予作文以记之。
予观夫巴陵胜状，在洞庭一湖。衔远山，吞长江，浩浩汤汤，横无际涯；朝晖夕阴，气象万千。此则岳阳楼之大观也，前人之述备矣。然则北通巫峡，南极潇湘，迁客骚人，多会于此，览物之情，得无异乎？
若夫霪雨霏霏，连月不开，阴风怒号，浊浪排空；日星隐曜，山岳潜形；商旅不行，樯倾楫摧；薄暮冥冥，虎啸猿啼。登斯楼也，则有去国怀乡，忧谗畏讥，满目萧然，感极而悲者矣。
至若春和景明，波澜不惊，上下天光，一碧万顷；沙鸥翔集，锦鳞游泳；岸芷汀兰，郁郁青青。而或长烟一空，皓月千里，浮光跃金，静影沉璧；渔歌互答，此乐何极！登斯楼也，则有心旷神怡，宠辱偕忘，把酒临风，其喜洋洋者矣。
嗟夫！予尝求古仁人之心，或异二者之为，何哉？不以物喜，不以己悲；居庙堂之高则忧其民，处江湖之远则忧其君。是进亦忧，退亦忧。然则何时而乐耶？其必曰“先天下之忧而忧，后天下之乐而乐”乎！噫！微斯人，吾谁与归？
时六年九月十五日。
"""

results = translator(text, max_new_tokens=1000)

print(results[0]["translation_text"])

In [ ]:
from transformers import pipeline

unmasker = pipeline(
    "fill-mask",
    model="distilbert/distilroberta-base",
    device="mps"
)

text = "Artificial intelligence is changing the <mask>."

results = unmasker(text, top_k=5)

for result in results:
    print(
        f"{result['token_str']:15} "
        f"Score: {result['score']:.4f} "
        f"Sentence: {result['sequence']}"
    )

In [ ]:

from transformers import pipeline

# Create an image-classification pipeline
classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

# Classify an image from a URL
image_url = (
    "https://huggingface.co/datasets/"
    "huggingface/documentation-images/resolve/main/coco_sample.png"
)

results = classifier(image_url)

# Print the predictions
for result in results[:5]:
    print(
        f"Label: {result['label']}, "
        f"Score: {result['score']:.4f}"
    )


In [ ]:
%pip install timm

In [ ]:

from transformers import pipeline
from PIL import Image

detector = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
    device="mps"
)

image = Image.open("cat.jpeg").convert("RGB")

results = detector(image)

for result in results:
    print(
        f"{result['label']:15} "
        f"Score: {result['score']:.4f} "
        f"Box: {result['box']}"
    )


In [ ]:
%pip install gTTS

In [ ]:

from gtts import gTTS

text = "Artificial intelligence is changing the way we work."

tts = gTTS(
    text=text,
    lang="en"
)

tts.save("speech.mp3")

print("Speech saved to speech.mp3")


In [ ]:
%pip install imageio-ffmpeg

In [ ]:
!python -c "import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"

In [ ]:

from transformers import pipeline

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device="mps"
)

audio = "speech.mp3"

result = transcriber(audio)

print(f"Transcription: {result['text']}")


In [ ]:

from gtts import gTTS

tts = gTTS(
    text="人工智能正在改变我们的生活。",
    lang="zh-CN"
)
tts.save("chinese_speech.mp3")

In [ ]:
result = transcriber("chinese_speech.mp3")

print(f"Transcription: {result['text']}")

In [ ]:
result = transcriber("speech.mp3")

print(f"Transcription: {result['text']}")

In [ ]:

from transformers import pipeline
import soundfile as sf

# Create a text-to-speech pipeline
synthesizer = pipeline(
    "text-to-speech",
    model="suno/bark-small",
    device="mps"
)

# Text to convert into speech
text = "Hello! Welcome to my artificial intelligence project."

# Generate speech
result = synthesizer(text)

# Save generated audio to a WAV file
sf.write(
    "speech.wav",
    result["audio"].squeeze(),
    result["sampling_rate"]
)

print("Speech saved to speech.wav")
print("Sampling rate:", result["sampling_rate"], "Hz")


In [ ]:
question="What are Hugging Face pipelines?"
context="Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device="mps")
result = question_answerer(question=question, context=context)
print(result)

In [ ]:
summarizer = pipeline("summarization", device="mps")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

In [ ]:
translator = pipeline("translation_en_to_fr", device="mps")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Another translation, showing a model being specified
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="mps")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Classification

classifier = pipeline("zero-shot-classification", device="mps")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:
# Text Generation

generator = pipeline("text-generation", device="mps")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
# Image Generation - remember this?! Now you know what's going on
# Pipelines can be used for diffusion models as well as transformers

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("mps")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation

from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='mps')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])